# 10 — Model 2: XAI + CMI (faithfulness pilot)

The first rung of the faithfulness phase, and the **pilot** — every mechanic is written inline so the pipeline
is legible for the methods chapter: load a checkpoint → build the region grid → compute per-region
attributions → build MoRF/LeRF deletion curves → DDS / PES / CMI, plus concentration (the confound control).
Settings are the phase-level logged decisions (DECISIONS_LOG: n_samples=8000, zero PM, predicted-class target,
per-method device split, N=500, 5 seeds); see also `notebooks/methodology/00_methodology_checks.ipynb` for the
evidence behind them.

**No ground-truth recovery check.** Model 1 had readable coefficients; Models 2–5 do not, by design. **CMI,
with concentration as the confound control, is the headline** — not a recovery verdict.

**Run order (deliberate — cheap first, KernelSHAP last).** §1–§8 run in ~7 min: FeatureAblation (§3) and
Integrated Gradients (§4) attributions, saved (§5), then their CMI (§6), concentration (§7) and presentation
(§8). Then start **§9 (KernelSHAP, ~56 min)** and interpret §8 while it runs; §9 saves per-seed so an
interruption loses at most one seed. §10–§11 finish once §9 is done.

> **You run the cells.** Every cell over ~1 min prints a time estimate up front.

## §1. Setup, settings, paths (all parameters defined once here)

In [ ]:
import sys, json, time
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import sleep_edf.config as cfg
from sleep_edf.loader import load_sleep_edf
from harness.models.cnn import build_cnn, torch_predict_proba
from harness.xai.regions import build_region_grid
from harness.xai.feature_ablation import feature_ablation          # perturbation-based (black-box)
from harness.xai.kernel_shap import kernel_shap                    # Shapley (black-box), batched
from harness.xai.integrated_gradients import integrated_gradients  # gradient-based (needs the torch model)
from harness.xai.deletion_curves import perturbation_curves        # MoRF / LeRF curves
from harness.xai.cmi import compute_cmi                            # DDS -> PES -> CMI
from harness.xai.concentration import region_concentration         # confound control (model-level reliance)

# ── Phase settings (logged decisions; do not vary across the ladder) ──────────
CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SEEDS       = [0, 1, 2, 3, 4]        # per-seed attribution + CMI, aggregated mean ± std
PM          = "zero"                 # perturbation method (§5 methodology): uniform hiding on z-scored EEG
N_EVAL      = 500                    # evaluation samples, stratified by class (bootstrap: CMI std 0.006 @ N=500)
EVAL_SEED   = 42
KS_N_SAMPLES = 8000                  # KernelSHAP coalitions (methodology §1: converged-enough, biased low)
KS_PE        = 200                   # perturbations_per_eval — batches coalitions (MPS win; equivalence 8e-8)
# target class = PREDICTED (argmax on the unperturbed signal). The harness default target_class=None IS
# this rule, so we pass None throughout; the attribution and its deletion curves then track the same class.

# ── Device policy (methodology §6, measured) ─────────────────────────────────
MPS = "mps" if torch.backends.mps.is_available() else "cpu"
DEV_FA   = "cpu"     # FeatureAblation: 50 tiny batch-1 forwards -> CPU fastest
DEV_IG   = MPS       # Integrated Gradients: one batched fwd+bwd -> MPS
DEV_KS   = MPS       # KernelSHAP batched (perturbations_per_eval) -> MPS
DEV_DEL  = "cpu"     # deletion curves / concentration: tiny batch-1 forwards -> CPU
if MPS == "cpu": KS_PE = 1; print("!! MPS unavailable — KS falls back to CPU, perturbations_per_eval=1 (slower).")

CKPT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "checkpoints"
OUT_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "metrics";  OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "figures";  FIG_DIR.mkdir(parents=True, exist_ok=True)

grid = build_region_grid(cfg.INPUT_LENGTH, cfg.REGION_SIZE_PRIMARY_PCT)   # 3000 -> 50 regions of 60 (0.6 s each)
SEC_PER_REGION = int(np.unique(grid.sizes)[0]) / cfg.SAMPLING_RATE

def load_model2(seed, device):
    """Load Model 2 (shallow CNN, kernel 15) seed `seed`, eval mode, on `device`."""
    m = build_cnn("shallow", n_classes=cfg.N_CLASSES, in_channels=cfg.IN_CHANNELS, kernel_size=cfg.CNN_KERNEL_SIZE)
    m.load_state_dict(torch.load(CKPT_DIR / f"model2_shallow_cnn_seed{seed}.pt", map_location="cpu")); m.eval()
    return m.to(device)

def stratified_idx(y, per_class, seed):
    rng = np.random.RandomState(seed); out = []
    for c in range(cfg.N_CLASSES): out += list(rng.choice(np.where(y == c)[0], per_class, replace=False))
    return np.array(out)

print(f"grid: {grid.n_regions} regions x {int(np.unique(grid.sizes)[0])} samples ({SEC_PER_REGION:.1f} s each) | "
      f"MPS={MPS}")
print(f"settings: PM={PM} | target=PREDICTED | N_EVAL={N_EVAL} | seeds={SEEDS} | KS n_samples={KS_N_SAMPLES}, pe={KS_PE}")
print(f"device: FA={DEV_FA} IG={DEV_IG} KS={DEV_KS} deletion/concentration={DEV_DEL}")

## §2. Evaluation subset — build once, save, reuse (fixed ladder-wide)

`N = 500` test epochs stratified by TRUE class (100/class), seed 42. Built once and saved; loaded thereafter,
never regenerated — **shared across all 5 seeds and all four models**, for the same reason the training
subsample and validation split are fixed: evaluating models on different samples would confound the ladder
comparison. Also reports, per seed, how many of the 500 Model 2 misclassifies and their class breakdown —
because the predicted-class rule means those samples attribute toward the (wrong) predicted class.

In [ ]:
EVAL_IDX_PATH = OUT_DIR / "xai_eval_subset_idx.npy"
X_test, y_test = load_sleep_edf("test", verbose=False)
if EVAL_IDX_PATH.exists():
    eval_idx = np.load(EVAL_IDX_PATH); print("loaded fixed eval subset:", EVAL_IDX_PATH.name)
else:
    eval_idx = stratified_idx(y_test, N_EVAL // cfg.N_CLASSES, EVAL_SEED)
    np.save(EVAL_IDX_PATH, eval_idx)
    json.dump({"n": int(len(eval_idx)), "per_class": N_EVAL // cfg.N_CLASSES, "seed": EVAL_SEED, "source": "test"},
              open(OUT_DIR / "xai_eval_subset_meta.json", "w"), indent=2)
    print("BUILT + saved fixed eval subset:", EVAL_IDX_PATH.name)

eval_sigs = X_test[eval_idx].astype(float)            # (500, 3000)
eval_true = y_test[eval_idx]                          # true labels (for stratification / diagnostics only)
assert len(eval_idx) == N_EVAL and set(np.bincount(eval_true)) == {N_EVAL // cfg.N_CLASSES}
print("per-class counts (true):", {CLASS_NAMES[c]: int((eval_true == c).sum()) for c in range(cfg.N_CLASSES)})

# Misclassification per seed (predicted != true). Attributions track the PREDICTED class regardless.
print(f"\n{'seed':>5}{'misclassified':>15}   breakdown (true->pred counts)")
for seed in SEEDS:
    pp = torch_predict_proba(load_model2(seed, DEV_FA), device=DEV_FA)
    pred = pp(eval_sigs).argmax(1)
    mis = int((pred != eval_true).sum())
    br = {}
    for t, p in zip(eval_true[pred != eval_true], pred[pred != eval_true]):
        br[f"{CLASS_NAMES[t]}->{CLASS_NAMES[p]}"] = br.get(f"{CLASS_NAMES[t]}->{CLASS_NAMES[p]}", 0) + 1
    top = dict(sorted(br.items(), key=lambda kv: -kv[1])[:6])
    print(f"{seed:>5}{mis:>15}   {top}")

## §3. FeatureAblation attributions — all 5 seeds  (~2 min, CPU)

FeatureAblation hides one region at a time (zero baseline) and records the drop in the predicted class's
probability = the per-region reliance. It is the **validation control** — with a zero baseline it computes
exactly the oracle quantity the deletion curves reward, so it should score high by construction. Result:
`fa_attr` of shape (5 seeds, 500 samples, 50 regions).

In [ ]:
fa_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"FeatureAblation: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_FA}. Estimated ~2 min.", flush=True)
for si, seed in enumerate(SEEDS):
    pp = torch_predict_proba(load_model2(seed, DEV_FA), device=DEV_FA)     # CPU predict_proba
    t0 = time.perf_counter()
    for j in range(N_EVAL):
        fa_attr[si, j] = feature_ablation(pp, eval_sigs[j], grid, PM)      # target_class=None -> predicted
    dt = time.perf_counter() - t0
    if si == 0:
        print(f"  seed {seed}: {dt:.0f}s  ->  est ~{dt*len(SEEDS)/60:.1f} min total (continues automatically)", flush=True)
    else:
        print(f"  seed {seed}: {dt:.0f}s", flush=True)
print("FeatureAblation done. fa_attr:", fa_attr.shape)

## §4. Integrated Gradients attributions — all 5 seeds  (~1 min, MPS)

IG integrates the model's gradient along a straight path from the zero baseline to the input, summed per
region (completeness). Gradient-based, so it takes the torch model directly (not `predict_proba`). Runs on
MPS. Result: `ig_attr` (5, 500, 50).

In [ ]:
ig_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"Integrated Gradients: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_IG}. Estimated ~1 min.", flush=True)
for si, seed in enumerate(SEEDS):
    model = load_model2(seed, DEV_IG)
    if si == 0:                                                            # warm up MPS (first calls compile)
        for _ in range(3): integrated_gradients(model, eval_sigs[0], grid, PM)
    t0 = time.perf_counter()
    for j in range(N_EVAL):
        ig_attr[si, j] = integrated_gradients(model, eval_sigs[j], grid, PM)   # target None -> predicted
    dt = time.perf_counter() - t0
    if si == 0:
        print(f"  seed {seed}: {dt:.0f}s  ->  est ~{dt*len(SEEDS)/60:.1f} min total", flush=True)
    else:
        print(f"  seed {seed}: {dt:.0f}s", flush=True)
print("Integrated Gradients done. ig_attr:", ig_attr.shape)

## §5. Save §3 + §4 attributions to disk  (before KernelSHAP)

Persist FA and IG attributions + the eval indices, so §6–§8 can run (and re-run) without recomputing, and so
starting the long §9 never risks the cheap results.

In [ ]:
np.savez(OUT_DIR / "model2_xai_fa_ig_attr.npz",
         fa_attr=fa_attr, ig_attr=ig_attr, eval_idx=eval_idx, seeds=np.array(SEEDS))
print("saved:", (OUT_DIR / "model2_xai_fa_ig_attr.npz").name)

## §6. CMI for FeatureAblation and IG  (~3 min, CPU)

The mechanic, shown inline. For each seed and each sample: `perturbation_curves` orders the 50 regions by the
method's attribution and builds two cumulative curves — **MoRF** (hide most-relevant first; should fall fast)
and **LeRF** (least-relevant first; should stay high). `compute_cmi` turns the per-sample (MoRF, LeRF) pairs
into **DDS** (per-sample front-weighted gap), **PES** (sign-consistency of DDS across samples), and **CMI**
(harmonic mean of |mean DDS| and |PES|). We aggregate CMI/DDS/PES to mean ± std across the 5 seeds.

In [ ]:
def cmi_for(attr, label, est_min):
    """Per-seed CMI/DDS/PES for an attribution array (n_seeds,N,50). Deletion curves on CPU (predicted class)."""
    print(f"{label}: deletion curves for {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_DEL}. Est ~{est_min} min.", flush=True)
    per_seed = []
    for si, seed in enumerate(SEEDS):
        pp = torch_predict_proba(load_model2(seed, DEV_DEL), device=DEV_DEL)
        M, L = [], []
        t0 = time.perf_counter()
        for j in range(N_EVAL):
            c = perturbation_curves(pp, eval_sigs[j], grid, attr[si, j], method=PM)  # target None -> predicted
            M.append(c["MoRF"]); L.append(c["LeRF"])
        r = compute_cmi(M, L)                              # {CMI, DDS, PES, dds_per_sample}
        per_seed.append({"seed": seed, "CMI": r["CMI"], "DDS": r["DDS"], "PES": r["PES"]})
        if si == 0: print(f"  seed {seed}: {time.perf_counter()-t0:.0f}s -> est ~{(time.perf_counter()-t0)*len(SEEDS)/60:.1f} min", flush=True)
    return per_seed

def agg(per_seed, key):
    v = np.array([d[key] for d in per_seed]); return v.mean(), v.std()

def ms(per_seed, key):
    m, s = agg(per_seed, key); return f"{m:.3f}±{s:.3f}"     # "mean±std" string for tables

fa_cmi = cmi_for(fa_attr, "FeatureAblation", 1.5)
ig_cmi = cmi_for(ig_attr, "IntegratedGradients", 1.5)
for name, ps in [("FeatureAblation", fa_cmi), ("IntegratedGradients", ig_cmi)]:
    print(f"\n{name}:  CMI {agg(ps,'CMI')[0]:.3f} ± {agg(ps,'CMI')[1]:.3f} | "
          f"DDS {agg(ps,'DDS')[0]:.3f} ± {agg(ps,'DDS')[1]:.3f} | PES {agg(ps,'PES')[0]:.3f} ± {agg(ps,'PES')[1]:.3f}")
    print("   per-seed PES:", [round(d["PES"], 3) for d in ps])

## §7. Concentration — the confound control  (~1.5 min, CPU)

Concentration is a property of the **model**, not any attribution: for each sample it hides each region singly,
forms the per-region reliance, and reports `1 − normalised entropy` (0 = diffuse, 1 = all reliance in one
region). When CMI moves across the ladder, concentration says whether that is genuine faithfulness change or a
mechanical consequence of the model relying on fewer regions. Reported mean ± std across seeds.

In [ ]:
print(f"Concentration: {len(SEEDS)} seeds x {N_EVAL} samples on {DEV_DEL}. Estimated ~1.5 min.", flush=True)
conc_per_seed = []
for si, seed in enumerate(SEEDS):
    pp = torch_predict_proba(load_model2(seed, DEV_DEL), device=DEV_DEL)
    t0 = time.perf_counter()
    vals = np.array([region_concentration(pp, eval_sigs[j], grid, PM) for j in range(N_EVAL)])  # predicted class
    conc_per_seed.append(float(np.nanmean(vals)))
    if si == 0: print(f"  seed {seed}: {time.perf_counter()-t0:.0f}s -> est ~{(time.perf_counter()-t0)*len(SEEDS)/60:.1f} min", flush=True)
concentration_mean, concentration_std = float(np.mean(conc_per_seed)), float(np.std(conc_per_seed))
print(f"\nconcentration: {concentration_mean:.3f} ± {concentration_std:.3f}   per-seed: {[round(c,3) for c in conc_per_seed]}")

## §8. Presentation — FeatureAblation vs IG (before KernelSHAP)

Attribution heatmaps over the 30-second epoch (per predicted class × region, mean over seeds), the CMI/DDS/PES
table for the two methods, and their **cross-method rank agreement** — the within-model half of the
hypothesis's logic: two methods agreeing points to a property of the *model*; one diverging points to a
property of the *method*. (Interpret these while §9/KernelSHAP runs.)

In [ ]:
def per_class_heatmap(attr, seed_preds):
    """Mean attribution per (predicted class x region), averaged over seeds. attr:(S,N,50)."""
    H = np.full((cfg.N_CLASSES, grid.n_regions), np.nan)
    for c in range(cfg.N_CLASSES):
        rows = [attr[si][seed_preds[si] == c].mean(0) for si in range(len(SEEDS)) if (seed_preds[si] == c).any()]
        if rows: H[c] = np.mean(rows, axis=0)
    return H

seed_preds = np.array([torch_predict_proba(load_model2(s, DEV_FA), device=DEV_FA)(eval_sigs).argmax(1) for s in SEEDS])
extent = [0, grid.n_regions * SEC_PER_REGION, cfg.N_CLASSES - 0.5, -0.5]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, attr, name in [(ax[0], fa_attr, "FeatureAblation"), (ax[1], ig_attr, "IntegratedGradients")]:
    H = per_class_heatmap(attr, seed_preds)
    v = np.nanmax(np.abs(H))
    im = a.imshow(H, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, extent=extent)
    a.set_yticks(range(cfg.N_CLASSES)); a.set_yticklabels(CLASS_NAMES); a.set_xlabel("time within epoch (s)")
    a.set_title(f"{name}: mean attribution (predicted class x region)"); fig.colorbar(im, ax=a, fraction=.04)
ax[0].set_ylabel("predicted stage")
fig.suptitle("Model 2 — per-region attribution over the 30 s epoch (mean over 5 seeds)", y=1.03)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_10_model2_xai_heatmaps_fa_ig.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'method':<20}{'CMI':>16}{'DDS':>16}{'PES':>16}")
for name, ps in [("FeatureAblation", fa_cmi), ("IntegratedGradients", ig_cmi)]:
    print(f"{name:<20}{ms(ps,'CMI'):>16}{ms(ps,'DDS'):>16}{ms(ps,'PES'):>16}")

# Cross-method rank agreement (Spearman over the 50 regions), per sample, mean over samples then seeds.
xm = np.mean([[spearmanr(fa_attr[si, j], ig_attr[si, j]).correlation for j in range(N_EVAL)] for si in range(len(SEEDS))])
print(f"\nFA vs IG rank agreement (mean Spearman over 500 samples x 5 seeds): {xm:.3f}")
print("high -> the two methods largely agree on region ordering (a model property); low -> method-dependent.")

## §9. ▶ KernelSHAP attributions — all 5 seeds  (~56 min, MPS, saves per seed)

**The long one — start it and walk away.** KernelSHAP estimates each region's Shapley value from sampled
coalitions (`n_samples=8000`), batched onto MPS via `perturbations_per_eval=200`. It is the primary
model-agnostic method carried across the whole ladder. Each seed is **saved to disk as it finishes**
(`model2_xai_ks_attr_seed{seed}.npy`), so an interruption loses at most one seed; re-running skips seeds
already saved. Expected ~11 min/seed → ~56 min total (seed 0 prints the measured projection).

In [ ]:
ks_attr = np.full((len(SEEDS), N_EVAL, grid.n_regions), np.nan)
print(f"KernelSHAP: {len(SEEDS)} seeds x {N_EVAL} samples, n_samples={KS_N_SAMPLES}, pe={KS_PE}, on {DEV_KS}.")
print(f"Estimated ~56 min total (~11 min/seed). Per-seed files are saved as each completes.", flush=True)
for si, seed in enumerate(SEEDS):
    seed_path = OUT_DIR / f"model2_xai_ks_attr_seed{seed}.npy"
    if seed_path.exists():
        ks_attr[si] = np.load(seed_path); print(f"  seed {seed}: loaded from disk ({seed_path.name})", flush=True); continue
    pp = torch_predict_proba(load_model2(seed, DEV_KS), device=DEV_KS)
    if si == 0:                                                    # warm MPS
        kernel_shap(pp, eval_sigs[0], grid, PM, n_samples=KS_N_SAMPLES, perturbations_per_eval=KS_PE)
    t0 = time.perf_counter()
    for j in range(N_EVAL):
        ks_attr[si, j] = kernel_shap(pp, eval_sigs[j], grid, PM, n_samples=KS_N_SAMPLES,
                                     target_class=None, seed=0, perturbations_per_eval=KS_PE)  # predicted class
    np.save(seed_path, ks_attr[si])                                # SAVE per seed immediately
    dt = time.perf_counter() - t0
    if si == 0: print(f"  seed {seed}: {dt/60:.1f} min -> est ~{dt*len(SEEDS)/60:.0f} min total. Saved {seed_path.name}.", flush=True)
    else:       print(f"  seed {seed}: {dt/60:.1f} min. Saved {seed_path.name}.", flush=True)
print("KernelSHAP done. ks_attr:", ks_attr.shape)

## §10. CMI for KernelSHAP + dip-check numbers  (~2 min)

CMI/DDS/PES for KernelSHAP (same deletion-curve mechanic as §6), then the **dip-check** numbers per the logged
policy: KernelSHAP top-25 **set overlap** and top-10 **order stability** between `n_samples` 4000 and 8000 on a
small stratified subsample. For Model 2 (the first rung) there is no neighbour to dip against yet — these are
computed and recorded so the cross-ladder dip-check (any rung whose CMI dips below its neighbours) can use
them. A high, stable CMI needs no such check; the check exists only where under-sampling and genuine
unfaithfulness are confusable (a downward dip).

In [ ]:
ks_cmi = cmi_for(ks_attr, "KernelSHAP", 1.5)
print(f"\nKernelSHAP:  CMI {agg(ks_cmi,'CMI')[0]:.3f} ± {agg(ks_cmi,'CMI')[1]:.3f} | "
      f"DDS {agg(ks_cmi,'DDS')[0]:.3f} ± {agg(ks_cmi,'DDS')[1]:.3f} | PES {agg(ks_cmi,'PES')[0]:.3f} ± {agg(ks_cmi,'PES')[1]:.3f}")

# Dip-check: KS stability between n_samples 4000 and 8000 on a 25-sample stratified subsample (seed 0 model).
sub = stratified_idx(eval_true, 5, EVAL_SEED)          # 25 of the 500 (indices into the eval subset), 5/class
pp0 = torch_predict_proba(load_model2(0, DEV_KS), device=DEV_KS)
a8 = ks_attr[0, sub]                                    # already have n=8000 for seed 0
print(f"dip-check: KS n=4000 on {len(sub)} samples (seed 0) for the 4000->8000 comparison ~20s ...", flush=True)
a4 = np.array([kernel_shap(pp0, eval_sigs[k], grid, PM, n_samples=4000, seed=0, perturbations_per_eval=KS_PE) for k in sub])
def topk(a, b, k):
    ta, tb = set(np.argsort(a)[::-1][:k]), set(np.argsort(b)[::-1][:k]); ix = list(tb)
    return len(ta & tb)/k, (spearmanr(a[ix], b[ix]).correlation if k > 1 else np.nan)
ov25 = np.mean([topk(a4[i], a8[i], 25)[0] for i in range(len(sub))])
or10 = np.mean([topk(a4[i], a8[i], 10)[1] for i in range(len(sub))])
full = np.mean([spearmanr(a4[i], a8[i]).correlation for i in range(len(sub))])
print(f"KS 4000->8000 (n={len(sub)}): full Spearman {full:.3f} | top-25 set overlap {ov25:.3f} | top-10 order {or10:.3f}")
print("(recorded for the cross-ladder dip-check; not a pass/fail on its own for the top rung.)")

## §11. Full presentation — all three methods + verdict

All three methods together: CMI/DDS/PES (mean ± std over seeds) with **PES surfaced explicitly** (Model 1's
PES = 1.0 was a linear-model artifact — whether it drops for this nonlinear CNN is a named watch-thread), the
three attribution heatmaps, the **three-way cross-method rank agreement**, and a written verdict. The headline
is CMI read alongside concentration (the confound control) — not a ground-truth recovery.

In [ ]:
methods = [("FeatureAblation", fa_attr, fa_cmi), ("KernelSHAP", ks_attr, ks_cmi), ("IntegratedGradients", ig_attr, ig_cmi)]

# CMI/DDS/PES table + explicit per-seed PES
print(f"{'method':<20}{'CMI':>16}{'DDS':>16}{'PES (mean±std)':>18}   per-seed PES")
for name, _, ps in methods:
    print(f"{name:<20}{ms(ps,'CMI'):>16}{ms(ps,'DDS'):>16}{ms(ps,'PES'):>18}   {[round(d['PES'],3) for d in ps]}")
print(f"\nconcentration (confound control): {concentration_mean:.3f} ± {concentration_std:.3f}")
# PES watch-thread -- distinguish SATURATION (at the ceiling) from a GENUINE drop, not "any value < 1.0".
pes_vals = [d["PES"] for _, _, ps in methods for d in ps]
pes_min = min(pes_vals)
if pes_min >= 0.97:
    print(f"PES watch-thread: SATURATED (all method/seed PES >= 0.97; min {pes_min:.3f}, most = 1.000). "
          "CMI is the harmonic mean of DDS and PES, so with PES at the ceiling CMI carries no information "
          "beyond DDS -- the consistency component is NOT discriminating in this setting.")
elif pes_min < 0.9:
    print(f"PES watch-thread: GENUINE DROP -- PES falls to {pes_min:.3f} (< 0.9) for some method/seed, so "
          "the consistency component IS discriminating here. Report which method/seed.")
else:
    print(f"PES watch-thread: near-ceiling (min {pes_min:.3f}, in [0.9, 0.97)) -- mostly saturated with some "
          "single-sample variation, not a break from the ceiling; CMI still tracks DDS closely.")

# Three attribution heatmaps
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (name, attr, _) in zip(ax, methods):
    H = per_class_heatmap(attr, seed_preds); v = np.nanmax(np.abs(H))
    im = a.imshow(H, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, extent=extent)
    a.set_yticks(range(cfg.N_CLASSES)); a.set_yticklabels(CLASS_NAMES); a.set_xlabel("time (s)"); a.set_title(name)
    fig.colorbar(im, ax=a, fraction=.04)
ax[0].set_ylabel("predicted stage")
fig.suptitle("Model 2 — attribution over the epoch, all three methods (mean over 5 seeds)", y=1.03)
fig.tight_layout(); fig.savefig(FIG_DIR / "sleep_edf_10_model2_xai_heatmaps_all.png", dpi=150, bbox_inches="tight")
plt.show()

# Three-way cross-method rank agreement (mean Spearman over samples x seeds)
def pair_agree(A, B): return np.mean([[spearmanr(A[si,j], B[si,j]).correlation for j in range(N_EVAL)] for si in range(len(SEEDS))])
print(f"\ncross-method rank agreement (mean Spearman):")
print(f"  FA–KS {pair_agree(fa_attr, ks_attr):.3f} | FA–IG {pair_agree(fa_attr, ig_attr):.3f} | KS–IG {pair_agree(ks_attr, ig_attr):.3f}")
print("  methods agreeing -> a property of the MODEL; one diverging -> a property of that METHOD.")

# Save the numeric results for the ladder-level comparison later.
results = {"model": "model2_shallow_cnn", "n_eval": N_EVAL, "eval_seed": EVAL_SEED, "pm": PM,
           "ks_n_samples": KS_N_SAMPLES, "concentration": {"mean": concentration_mean, "std": concentration_std},
           "methods": {name: {"per_seed": ps,
                              "CMI_mean": agg(ps,"CMI")[0], "CMI_std": agg(ps,"CMI")[1],
                              "DDS_mean": agg(ps,"DDS")[0], "PES_mean": agg(ps,"PES")[0]}
                       for name, _, ps in methods}}
json.dump(results, open(OUT_DIR / "model2_xai_cmi_results.json", "w"), indent=2, default=float)
print("\nsaved:", (OUT_DIR / "model2_xai_cmi_results.json").name)

### Verdict (fill in after running)

- **CMI per method** (mean ± std), read against **concentration** — is Model 2's faithfulness high/low, and is
  that a genuine property or a concentration effect?
- **FeatureAblation** is the control (should be high by construction); **KernelSHAP** is the ladder
  through-line; **IG** the gradient cross-check.
- **PES watch-thread:** did PES stay 1.0 across all seeds, or drop for this nonlinear model?
- **Cross-method agreement:** do the three methods agree on region ordering (a model property) or does one
  diverge (a method property)?
- These are the numbers the ladder-level comparison (Models 2→5) will read against — with the dip-check
  applied to any rung whose CMI dips below its neighbours.